In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 2_presenca_eventos_transform
# LOCAL: /2_silver/src/
# OBJETIVO: Transformar a tabela de presenças em uma tabela Silver com os tipos de eventos
# ----------------------------------------------------------------------------------------

from pyspark.sql.functions import col

# Carregar as tabelas
df_presenca_bronze = spark.table("bronze_presenca_eventos")
df_eventos_silver = spark.table("silver_eventos") 

# Selecionar apenas o necessário da tabela de eventos para o JOIN
df_eventos_tipos = df_eventos_silver.select(
    col("id").alias("id_evento"), 
    col("descricaoTipo").alias("tipo_evento"),
    col("dataHoraInicio") # Útil para filtrar "ao longo do ano"
)

# Join para saber o tipo de cada evento onde o deputado esteve
df_presenca_silver = df_presenca_bronze.join(
    df_eventos_tipos, 
    on="id_evento", 
    how="inner"
)

# 1. Remova as duplicatas primeiro (Garanta que cada deputado apareça apenas uma vez por evento)
df_presenca_silver_clean = df_presenca_silver.dropDuplicates(["id", "id_evento"])

# Salvar na Silver
df_presenca_silver_clean.write.mode("overwrite").saveAsTable("silver_presenca_eventos")


display(spark.table("silver_presenca_eventos"))